# Overlap flow: how tile B sits relative to tile A, four ways

For every pair of overlapping tiles, the contacts both tiles see are matched and compared
(B − A, in nominal mask coordinates; the mask error cancels, so this is pure SEM placement error):

1. **Robust translation** (`fit_overlap`): least-squares shift; contacts whose residual is more
   than `OUTLIER_FACTOR` × the median residual are flagged as outliers and left out. The
   residual RMS of the rest is the **measurement noise floor**.
2. **Phase correlation** of the overlap pixels (`image_overlap_shift`): a shift that does not
   depend on contact detection at all. It should agree with step 1 within the noise.
3. **Translation + rotation** (`fit_overlap(..., rotation=True)`, no scale): keep rotation only if
   it lowers the RMS clearly and consistently across pairs.
4. **Residual map**: the step-1 residuals of all pairs drawn at their position in tile A.
   Random scatter means noise; a pattern that repeats at the same place points to SEM distortion.

Positions are shown in µm, shifts and residuals in nm, rotations in µrad.

In [ ]:
DATA_DIR = r"A:\Stitch_monitor\In_cell_without_PTM_20260914\260910_1143_Norm"
TILES = list(range(10))   # which tiles to use (indices in read_tile_index order)
REFINE_EDGES = True       # refine Otsu contours to the maximum intensity gradient
SEARCH_PX = 5.0           # refinement searches this many pixels in/out along the contour normal
MAX_MATCH_NM = 10.0       # pair contacts closer than this
OUTLIER_FACTOR = 3.0      # outlier if residual > this x the median residual
MIN_MATCHED = 5           # skip pairs with fewer matched contacts (e.g. corner-only overlaps)
PAIR = 0                  # which pair (row of the pair table) to look at in detail
ARROW_SCALE = 20          # residual arrows are drawn this many times longer

In [ ]:
%matplotlib widget
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from affine_ransac.features.contact import detect_contacts
from affine_ransac.features.edges import refine_edges
from affine_ransac.geometry.frames import pixel_to_tile_nm
from affine_ransac.io.metadata import read_tile_index
from affine_ransac.io.sem_image import load_sem_image
from affine_ransac.overlap import match_overlap, overlapping_pairs
from affine_ransac.overlap_fit import fit_overlap
from affine_ransac.overlap_image import image_overlap_shift

## Load the tiles and detect their contacts

In [ ]:
all_tiles = read_tile_index(DATA_DIR)
tiles = [all_tiles[i] for i in TILES]
centers = np.array([(t.center_x_nm, t.center_y_nm) for t in tiles])
fovs = np.array([(t.fov_x_nm, t.fov_y_nm) for t in tiles])
pixel_size_nm = tiles[0].fov_x_nm / tiles[0].image_width_px

start = time.perf_counter()
images, contacts = [], []   # per tile: the image, and its SEM contact centres in mask nm (nominal)
for tile, center in zip(tiles, centers):
    image = load_sem_image(tile.image_path)
    found = detect_contacts(image)
    if REFINE_EDGES:
        found = refine_edges(image, found, search_px=SEARCH_PX)
    images.append(image)
    contacts.append(pixel_to_tile_nm(found.centers, image.shape, pixel_size_nm) + center)
print(f"{len(tiles)} tiles, {sum(len(c) for c in contacts)} contacts, {time.perf_counter() - start:.1f} s")

## Overlapping pairs and their matched contacts

In [ ]:
pairs = []   # (i, j, box, matched A points, matched B points)
for i, j, box in overlapping_pairs(centers, fovs):
    ia, ib = match_overlap(contacts[i], contacts[j], box, MAX_MATCH_NM)
    if len(ia) >= MIN_MATCHED:
        pairs.append((i, j, box, contacts[i][ia], contacts[j][ib]))

pd.DataFrame([{
    "tile_a": tiles[i].tile_id, "tile_b": tiles[j].tile_id,
    "overlap_w_um": (box[1] - box[0]) / 1000, "overlap_h_um": (box[3] - box[2]) / 1000,
    "matched": len(a),
} for i, j, box, a, b in pairs])

## One pair in detail (`PAIR`)

### Step 1: robust translation

In [ ]:
i, j, box, a, b = pairs[PAIR]
fit_t = fit_overlap(a, b, outlier_factor=OUTLIER_FACTOR)
print(f"Pair {tiles[i].tile_id} -> {tiles[j].tile_id}: {len(a)} matched contacts, {(~fit_t.inliers).sum()} outliers")
print(f"Shift B - A: x {fit_t.shift[0]:.2f} nm, y {fit_t.shift[1]:.2f} nm")
print(f"Residual RMS (noise floor): {fit_t.rms():.2f} nm")


def residual_plot(a, fit, title):
    """Matched A contacts a with their residual arrows (magnified); outliers circled in red."""
    fig, ax = plt.subplots(figsize=(9, 4))
    x_um, y_um = a[:, 0] / 1000, a[:, 1] / 1000
    ax.quiver(x_um, y_um, fit.residuals[:, 0] / 1000 * ARROW_SCALE, fit.residuals[:, 1] / 1000 * ARROW_SCALE,
              angles="xy", scale_units="xy", scale=1, width=0.003)
    ax.scatter(x_um, y_um, marker="+", color="tab:blue", label="contact (tile A)")
    ax.scatter(x_um[~fit.inliers], y_um[~fit.inliers], s=150, facecolors="none", edgecolors="tab:red", label="outlier")
    ax.set_aspect("equal")
    ax.set_xlabel("mask x (µm)")
    ax.set_ylabel("mask y (µm)")
    ax.set_title(f"{title}: residuals x{ARROW_SCALE}, RMS {fit.rms():.2f} nm")
    ax.legend()
    plt.show()


residual_plot(a, fit_t, "Step 1, translation")

### Step 2: phase correlation of the overlap pixels

In [ ]:
image_shift, match_error = image_overlap_shift(images[i], centers[i], images[j], centers[j], pixel_size_nm, box)
print(f"Image shift B - A: x {image_shift[0]:.2f} nm, y {image_shift[1]:.2f} nm (match error {match_error:.2f}, lower is better)")
print(f"Image - contacts:  x {image_shift[0] - fit_t.shift[0]:+.2f} nm, y {image_shift[1] - fit_t.shift[1]:+.2f} nm")

### Step 3: translation + rotation

In [ ]:
fit_r = fit_overlap(a, b, rotation=True, outlier_factor=OUTLIER_FACTOR)
print(f"Rotation: {fit_r.rotation * 1e6:.1f} urad, shift x {fit_r.shift[0]:.2f} nm, y {fit_r.shift[1]:.2f} nm")
print(f"Residual RMS: {fit_r.rms():.2f} nm (translation only: {fit_t.rms():.2f} nm)")
residual_plot(a, fit_r, "Step 3, translation + rotation")

## Steps 1–3 for all pairs

`image_minus_contact`: how far phase correlation is from the contact-based shift.
`rms_rigid` vs `rms_translation`: how much rotation helps.

In [ ]:
rows, fits = [], []
for i, j, box, a, b in pairs:
    fit_t = fit_overlap(a, b, outlier_factor=OUTLIER_FACTOR)
    fit_r = fit_overlap(a, b, rotation=True, outlier_factor=OUTLIER_FACTOR)
    image_shift, match_error = image_overlap_shift(images[i], centers[i], images[j], centers[j], pixel_size_nm, box)
    fits.append((i, a, fit_t))
    rows.append({
        "tile_a": tiles[i].tile_id, "tile_b": tiles[j].tile_id, "matched": len(a), "outliers": int((~fit_t.inliers).sum()),
        "shift_x_nm": fit_t.shift[0], "shift_y_nm": fit_t.shift[1], "rms_translation_nm": fit_t.rms(),
        "image_minus_contact_x_nm": image_shift[0] - fit_t.shift[0],
        "image_minus_contact_y_nm": image_shift[1] - fit_t.shift[1], "match_error": match_error,
        "rotation_urad": fit_r.rotation * 1e6, "rms_rigid_nm": fit_r.rms(),
    })
summary = pd.DataFrame(rows)

print(f"{len(summary)} pairs")
print(f"Noise floor (median RMS, translation): {summary.rms_translation_nm.median():.2f} nm")
print(f"Median RMS with rotation:              {summary.rms_rigid_nm.median():.2f} nm")
print(f"Rotation: mean {summary.rotation_urad.mean():.1f} urad, std {summary.rotation_urad.std():.1f} urad")
print(f"Image - contact shift: x mean {summary.image_minus_contact_x_nm.mean():+.2f} nm (std {summary.image_minus_contact_x_nm.std():.2f}), "
      f"y mean {summary.image_minus_contact_y_nm.mean():+.2f} nm (std {summary.image_minus_contact_y_nm.std():.2f})")
print(f"Outliers: {summary.outliers.sum()} of {summary.matched.sum()} matched contacts")
summary

## Step 4: residuals of all pairs, at their position in tile A

Each arrow is one contact's step-1 residual (translation fit), placed where the contact is in
tile A's own frame (tile-local, µm). Outliers are circled.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
for i, a, fit in fits:
    local_um = (a - centers[i]) / 1000
    ax.quiver(local_um[:, 0], local_um[:, 1], fit.residuals[:, 0] / 1000 * ARROW_SCALE, fit.residuals[:, 1] / 1000 * ARROW_SCALE,
              angles="xy", scale_units="xy", scale=1, width=0.002)
    outliers = local_um[~fit.inliers]
    ax.scatter(outliers[:, 0], outliers[:, 1], s=80, facecolors="none", edgecolors="tab:red")
half_um = fovs[0] / 2000
ax.add_patch(plt.Rectangle(-half_um, *(2 * half_um), fill=False, color="gray", linestyle="--"))
ax.set_aspect("equal")
ax.set_xlabel("tile-local x (µm)")
ax.set_ylabel("tile-local y (µm)")
ax.set_title(f"Step-1 residuals of {len(fits)} pairs, x{ARROW_SCALE}")
plt.show()